---
# **Cuaderno 1:** Mutaciones en Cáncer
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: Agosto 2025

**Aims:**

* Obtiene Dataframe con mutaciones oncogenicas de 18 tejidos, y filtra por genes (necesita UniprotID)

* El "Consorcio Internacional del Genoma de Cáncer" (ICGC) es una organización científica voluntaria que proporciona un foro para colaboración entre los principales investigadores genómicos del mundo del cáncer, la cual se utilizó como base de datos durante este proyecto


* Se utilizarón y analizarón las mutaciones de los tejidos: Piel, hígado, sangre, cerebro, seno, ovario, páncreas, próstata, útero, estómago, pulmón, riñón, cabeza, vesícula, colon, cérvix, hueso y vejiga
    

### Librerias

In [1]:
import requests
import pandas as pd
from pyensembl import EnsemblRelease 

### Datos de ingreso

In [ ]:
#Nombre grupo de genes
grupo = "kit"
#Insetar código de uniprot
uniprot_ids = ["P10721", "P16234", "P09619", "P36888", "P07333"]


## 1. Obtener Nombre de gen y Ensembl ID


In [2]:

#Función para obtener UniprotID

def genes_unirpot (uniprot_ids):

    data = []  
    
    for uniprot_id in uniprot_ids: 
        response = requests.get(f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.json')
    
        if response.status_code == 200:
            json_data = response.json()
            gen_name = json_data['genes'][0]['geneName']['value']  # Extrae el nombre del gen
            
            # Guarda los datos en la lista
            data.append({"uniprot_id": uniprot_id, "gen_name": gen_name}) 
        else:
            print(f"Error al obtener {uniprot_id}: {response.status_code}")

    df = pd.DataFrame(data)
    
    return df

# Función para obtener EnsemblID

def agregar_id_ensembl(df):
    
    # La versión 109 (Febrero 2023) utiliza el genoma de referencia humano GRCh38
    data = EnsemblRelease(109)         
    ids_ensembl_gen = []
        # Itera sobre la columna 'nombre_gen'
    for nombre_gen in df['gen_name']: # Corregido de 'Gen_names' a 'nombre_gen'
        id_ensembl = None # Valor predeterminado a None
        if nombre_gen: 
            try:
                ids = data.gene_ids_of_gene_name(nombre_gen)
                if ids:
                    id_ensembl = ids[0]
                else:
                    print(f"No se encontró ID de Ensembl para el nombre de gen: {nombre_gen}")
            except Exception as e:
                # Captura cualquier error durante la búsqueda en pyensembl
                print(f"Error al buscar ID de Ensembl para el nombre de gen '{nombre_gen}': {e}")
        
        ids_ensembl_gen.append(id_ensembl)
                
    df['ensembl_id'] = ids_ensembl_gen 
    
    return df

# DataFrame con genes y ID's

df_genes = genes_unirpot(uniprot_ids)
df_genes = agregar_id_ensembl(df_genes.copy()) 
df_genes


,uniprot_id,gen_name,ensembl_id
0,P10721,KIT,ENSG00000157404
1,P16234,PDGFRA,ENSG00000134853
2,P09619,PDGFRB,ENSG00000113721
3,P36888,FLT3,ENSG00000122025
4,P07333,CSF1R,ENSG00000182578


## 2. Filtrar base de datos de ICGC 

Filtra la base de datos con los genes a utilizar. Para ello utiliza el EnsemblID y agrega las columnas gen_namey uniprot_id

In [3]:

#Ruta para guardar
ruta = "../data/Cancer/cancer_data_{0}.csv".format(grupo)

# Generar lista de ENS IDs únicos como set para búsqueda rápida
lista_ens_genes = set(df_genes['ensembl_id'].unique())

# Diccionario de archivos por tipo de cáncer
cancer_archivos = {
    
    'Piel': '../data/ICGC/icgc-dataset-skin/simple_somatic_mutation.open.tsv.gz',
    'Hígado': '../data/ICGC/icgc-dataset-liver/simple_somatic_mutation.open.tsv.gz',
    'Sangre': '../data/ICGC/icgc-dataset-blood/simple_somatic_mutation.open.tsv.gz',
    'Cerebro': '../data/ICGC/icgc-dataset-brain/simple_somatic_mutation.open.tsv.gz',
    'Seno': '../data/ICGC/icgc-dataset-breast/simple_somatic_mutation.open.tsv.gz',
    'Ovario': '../data/ICGC/icgc-dataset-ovario/simple_somatic_mutation.open.tsv.gz',
    'Páncreas': '../data/ICGC/icgc-dataset-pancreas/simple_somatic_mutation.open.tsv.gz',
    'Próstata': '../data/ICGC/icgc-dataset-prostate/simple_somatic_mutation.open.tsv.gz',
    'Útero': '../data/ICGC/icgc-dataset-uterus/simple_somatic_mutation.open.tsv.gz',
    'Estómago': '../data/ICGC/icgc-dataset-stomach/simple_somatic_mutation.open.tsv.gz',
    'Pulmón': '../data/ICGC/icgc-dataset-lung/simple_somatic_mutation.open.tsv.gz',
    'Riñón': '../data/ICGC/icgc-dataset-kidney/simple_somatic_mutation.open.tsv.gz',
    'Cabeza': '../data/ICGC/icgc-dataset-head/simple_somatic_mutation.open.tsv.gz',
    'Vesícula': '../data/ICGC/icgc-dataset-gladd/simple_somatic_mutation.open.tsv.gz',
    'Colon': '../data/ICGC/icgc-dataset-colonrectal/simple_somatic_mutation.open.tsv.gz',
    'Cérvix': '../data/ICGC/icgc-dataset-cervix/simple_somatic_mutation.open.tsv.gz',
    'Hueso': '../data/ICGC/icgc-dataset-bone/simple_somatic_mutation.open.tsv.gz',
    'Vejiga': '../data/ICGC/icgc-dataset-bladder/simple_somatic_mutation.open.tsv.gz'
}


chunk_size = 500_000
df_cancer = pd.DataFrame()

for idx, (tipo_cancer, archivo) in enumerate(cancer_archivos.items(), start=1):
    print(f"Procesando {idx}/{len(cancer_archivos)}: {tipo_cancer} ...")
    chunks = pd.read_csv(archivo, sep='\t', low_memory=False, chunksize=chunk_size)

    for chunk in chunks:
        merge_df = chunk[chunk['gene_affected'].isin(lista_ens_genes)].copy()
        merge_df['tipo_cancer'] = tipo_cancer
        df_cancer = pd.concat([df_cancer, merge_df], ignore_index=True)

    print(f"Completado: {tipo_cancer}")

print("Terminado")

#Merge para agregar nombre de gen

df_cancer = df_cancer.merge(df_genes, left_on="gene_affected", right_on="ensembl_id")

#Guardar archivo
df_cancer.to_csv(ruta, index=False)

df_cancer.head(2)

Procesando 1/18: Piel ...
Completado: Piel
Procesando 2/18: Hígado ...
Completado: Hígado
Procesando 3/18: Sangre ...
Completado: Sangre
Procesando 4/18: Cerebro ...
Completado: Cerebro
Procesando 5/18: Seno ...
Completado: Seno
Procesando 6/18: Ovario ...
Completado: Ovario
Procesando 7/18: Páncreas ...
Completado: Páncreas
Procesando 8/18: Próstata ...
Completado: Próstata
Procesando 9/18: Útero ...
Completado: Útero
Procesando 10/18: Estómago ...
Completado: Estómago
Procesando 11/18: Pulmón ...
Completado: Pulmón
Procesando 12/18: Riñón ...
Completado: Riñón
Procesando 13/18: Cabeza ...
Completado: Cabeza
Procesando 14/18: Vesícula ...
Completado: Vesícula
Procesando 15/18: Colon ...
Completado: Colon
Procesando 16/18: Cérvix ...
Completado: Cérvix
Procesando 17/18: Hueso ...
Completado: Hueso
Procesando 18/18: Vejiga ...
Completado: Vejiga
Terminado


,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,icgc_sample_id,matched_icgc_sample_id,submitted_sample_id,submitted_matched_sample_id,chromosome,chromosome_start,...,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,raw_data_repository,raw_data_accession,initial_data_release_date,tipo_cancer,uniprot_id,gen_name,ensembl_id
0,MU4360250,DO37659,SKCM-US,SP82900,SA432539,SA432575,TCGA-DA-A1HV-06A-21D-A196-08,TCGA-DA-A1HV-10A-01D-A198-08,4,55139809,...,PCAWG Consensus SNV-MNV caller,NaN,NaN,NaN,FI30697:FI30696,NaN,Piel,P16234,PDGFRA,ENSG00000134853
1,MU4360250,DO37659,SKCM-US,SP82900,SA432539,SA432575,TCGA-DA-A1HV-06A-21D-A196-08,TCGA-DA-A1HV-10A-01D-A198-08,4,55139809,...,PCAWG Consensus SNV-MNV caller,NaN,NaN,NaN,FI30697:FI30696,NaN,Piel,P16234,PDGFRA,ENSG00000134853


In [4]:
# Datos
print(f"tamaño: {df_cancer.shape}")
print(f"columnas: {df_cancer.columns}")
print(f"valores nulos: {df_cancer.isnull().sum()}")
total_size = tipo_cancer
print(f"")
df_cancer.head(2)

tamaño: (32689, 46)
columnas: Index(['icgc_mutation_id', 'icgc_donor_id', 'project_code', 'icgc_specimen_id',
       'icgc_sample_id', 'matched_icgc_sample_id', 'submitted_sample_id',
       'submitted_matched_sample_id', 'chromosome', 'chromosome_start',
       'chromosome_end', 'chromosome_strand', 'assembly_version',
       'mutation_type', 'reference_genome_allele', 'mutated_from_allele',
       'mutated_to_allele', 'quality_score', 'probability', 'total_read_count',
       'mutant_allele_read_count', 'verification_status',
       'verification_platform', 'biological_validation_status',
       'biological_validation_platform', 'consequence_type', 'aa_mutation',
       'cds_mutation', 'gene_affected', 'transcript_affected',
       'gene_build_version', 'platform', 'experimental_protocol',
       'sequencing_strategy', 'base_calling_algorithm', 'alignment_algorithm',
       'variation_calling_algorithm', 'other_analysis_algorithm',
       'seq_coverage', 'raw_data_repository', 'raw_d

,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,icgc_sample_id,matched_icgc_sample_id,submitted_sample_id,submitted_matched_sample_id,chromosome,chromosome_start,...,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,raw_data_repository,raw_data_accession,initial_data_release_date,tipo_cancer,uniprot_id,gen_name,ensembl_id
0,MU4360250,DO37659,SKCM-US,SP82900,SA432539,SA432575,TCGA-DA-A1HV-06A-21D-A196-08,TCGA-DA-A1HV-10A-01D-A198-08,4,55139809,...,PCAWG Consensus SNV-MNV caller,NaN,NaN,NaN,FI30697:FI30696,NaN,Piel,P16234,PDGFRA,ENSG00000134853
1,MU4360250,DO37659,SKCM-US,SP82900,SA432539,SA432575,TCGA-DA-A1HV-06A-21D-A196-08,TCGA-DA-A1HV-10A-01D-A198-08,4,55139809,...,PCAWG Consensus SNV-MNV caller,NaN,NaN,NaN,FI30697:FI30696,NaN,Piel,P16234,PDGFRA,ENSG00000134853


## 3. Seleccionar solo mutaciones

In [5]:
df_mut = df_cancer[["icgc_donor_id" ,"uniprot_id","gen_name", "aa_mutation","tipo_cancer", "transcript_affected","verification_status",  "consequence_type"]]
ruta = "../data/Cancer/mut_cancer_{0}.csv".format(grupo)
df_mut.to_csv(ruta, index=False)
df_mut.shape
df_mut


,icgc_donor_id,uniprot_id,gen_name,aa_mutation,tipo_cancer,transcript_affected,verification_status,consequence_type
0,DO37659,P16234,PDGFRA,NaN,Piel,ENST00000509092,not tested,exon_variant
1,DO37659,P16234,PDGFRA,NaN,Piel,ENST00000507536,not tested,upstream_gene_variant
2,DO37659,P16234,PDGFRA,NaN,Piel,ENST00000509490,not tested,exon_variant
3,DO37659,P16234,PDGFRA,F490F,Piel,ENST00000257290,not tested,synonymous_variant
4,DO37659,P16234,PDGFRA,NaN,Piel,ENST00000461294,not tested,upstream_gene_variant
...,...,...,...,...,...,...,...,...
32684,DO743,P16234,PDGFRA,NaN,Vejiga,ENST00000509092,not tested,exon_variant
32685,DO743,P16234,PDGFRA,NaN,Vejiga,ENST00000507536,not tested,upstream_gene_variant
32686,DO743,P16234,PDGFRA,R522H,Vejiga,ENST00000257290,not tested,missense_variant
32687,DO743,P16234,PDGFRA,NaN,Vejiga,ENST00000509490,not tested,exon_variant
